In [30]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns 
import numpy as np

In [31]:
rf_df = pd.read_csv('../results/rf_multiclass_k3_predictions.csv')
lr_df = pd.read_csv('../results/logreg_cluster_predictions.csv')
rf_group = pd.read_csv('../results/rf_group_predictions.csv')
lr_group = pd.read_csv('../results/logreg_group_predictions.csv')

    
# Merge on Sample/Sample ID (rename columns for consistency)
rf_df = rf_df.rename(columns={"sample_id": "Sample", "pred_cluster_rf": "rf_pred_cluster"})
lr_df = lr_df.rename(columns={"Pred_Cluster_LR": "lr_pred_cluster"})


In [32]:
# Merge all on sample ID
merged = rf_group[['Sample','Pred_Label_RF']].merge(
             lr_group[['Sample','Pred_Label_LR']], on='Sample'
         ).merge(
             rf_df[['Sample','rf_pred_cluster']], on='Sample'
         ).merge(
             lr_df[['Sample','lr_pred_cluster']], on='Sample'
         )

In [33]:
# For each sample, count group predictions (across both group models)
merged['group_preds'] = merged[['Pred_Label_RF', 'Pred_Label_LR']].values.tolist()
merged['group_stability'] = merged['group_preds'].apply(
    lambda preds: max([preds.count(x) for x in set(preds)]) / len(preds)
)

# For each sample, count cluster predictions (across both cluster models)
merged['cluster_preds'] = merged[['rf_pred_cluster', 'lr_pred_cluster']].values.tolist()
merged['cluster_stability'] = merged['cluster_preds'].apply(
    lambda preds: max([preds.count(x) for x in set(preds)]) / len(preds)
)

print(merged[['group_preds','group_stability','cluster_preds','cluster_stability']].head())


  group_preds  group_stability cluster_preds  cluster_stability
0    [T3, T3]              1.0        [1, 1]                1.0
1    [T0, T0]              1.0        [1, 1]                1.0
2    [T3, T3]              1.0        [1, 1]                1.0
3    [T0, T0]              1.0        [0, 0]                1.0


In [34]:
merged.head(10)

,Sample,Pred_Label_RF,Pred_Label_LR,rf_pred_cluster,lr_pred_cluster,group_preds,group_stability,cluster_preds,cluster_stability
0,SRR1785299,T3,T3,1,1,"[T3, T3]",1.0,"[1, 1]",1.0
1,SRR1785244,T0,T0,1,1,"[T0, T0]",1.0,"[1, 1]",1.0
2,SRR1785317,T3,T3,1,1,"[T3, T3]",1.0,"[1, 1]",1.0
3,SRR1785238,T0,T0,0,0,"[T0, T0]",1.0,"[0, 0]",1.0


In [35]:
from scipy.stats import pearsonr, spearmanr
from statsmodels.stats.multitest import multipletests


# Compute Pearson correlation and p-value
r, p = pearsonr(merged['group_stability'], merged['cluster_stability'])
print(f"Pearson r = {r:.3f}, p = {p:.3g}")

# Compute Spearman correlation and p-value
r_s, p_s = spearmanr(merged['group_stability'], merged['cluster_stability'])
print(f"Spearman rho = {r_s:.3f}, p = {p_s:.3g}")

# Collect p-values for multiple test correction
pvals = [p, p_s]

# Apply FDR correction (Benjamini-Hochberg)
reject, pvals_corrected, _, _ = multipletests(pvals, alpha=0.05, method='fdr_bh')

print(f"Corrected p-values: Pearson={pvals_corrected[0]:.3g}, Spearman={pvals_corrected[1]:.3g}")
print(f"Reject null hypotheses? Pearson={reject[0]}, Spearman={reject[1]}")

Pearson r = nan, p = nan
Spearman rho = nan, p = nan
Corrected p-values: Pearson=nan, Spearman=nan
Reject null hypotheses? Pearson=False, Spearman=False


/var/folders/sr/q_r5y_f53xj5n7_nx6m1qrw80000gn/T/ipykernel_73388/3685363086.py:6: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r, p = pearsonr(merged['group_stability'], merged['cluster_stability'])
/var/folders/sr/q_r5y_f53xj5n7_nx6m1qrw80000gn/T/ipykernel_73388/3685363086.py:10: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  r_s, p_s = spearmanr(merged['group_stability'], merged['cluster_stability'])
